In [15]:
import os
import pandas as pd
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

import os
os.chdir('E:\Backup Files Abhinav_Satyam\Practical Machine Learning\Sanjay Sir Git Repo\Datasets')


In [16]:
house = pd.read_csv('Housing.csv')
house

,price,lotsize,bedrooms,bathrms,stories,driveway,recroom,fullbase,gashw,airco,garagepl,prefarea
0,42000.0,5850,3,1,2,yes,no,yes,no,no,1,no
1,38500.0,4000,2,1,1,yes,no,no,no,no,0,no
2,49500.0,3060,3,1,1,yes,no,no,no,no,0,no
3,60500.0,6650,3,1,2,yes,yes,no,no,no,0,no
4,61000.0,6360,2,1,1,yes,no,no,no,no,0,no
...,...,...,...,...,...,...,...,...,...,...,...,...
541,91500.0,4800,3,2,4,yes,yes,no,no,yes,0,no
542,94000.0,6000,3,2,4,yes,no,no,no,yes,0,no
543,103000.0,6000,3,2,4,yes,yes,no,no,yes,1,no
544,105000.0,6000,3,2,2,yes,yes,no,no,yes,1,no


In [17]:
house.isna().sum()

price       0
lotsize     0
bedrooms    0
bathrms     0
stories     0
driveway    0
recroom     0
fullbase    0
gashw       0
airco       0
garagepl    0
prefarea    0
dtype: int64

In [22]:
house = pd.read_csv('Housing.csv')

X , y = house.drop('price' , axis = 1), house['price']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state=26)



ohe = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')
#2 imputer one for categorical for numerical
#and 2 ct, 1 for imputer and one for ohe

ct = make_column_transformer((ohe,make_column_selector(dtype_include=object)),
                            ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False).set_output(transform='pandas')

scaler = StandardScaler().set_output(transform='pandas')



ks = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
scores=[]
X_trans_train = ct.fit_transform(X_train)
X_scale_train = scaler.fit_transform(X_trans_train)

for k in ks:
    knn = KNeighborsRegressor(n_neighbors=k)
    
    knn.fit(X_scale_train , y_train)
    
    X_trans_test = ct.fit_transform(X_test)
    X_scale_test = scaler.fit_transform(X_trans_test)

    y_pred = knn.predict(X_scale_test) 
    scores.append([k,r2_score(y_test,y_pred)])

    
df_scores = pd.DataFrame(scores,columns=['K','score'])    
df_scores.sort_values('score',ascending=False)

,K,score
14,15,0.687634
11,12,0.678122
10,11,0.677219
9,10,0.675929
2,3,0.675607
5,6,0.674234
13,14,0.674114
12,13,0.674060
6,7,0.673477
7,8,0.671280


In [19]:
house = pd.read_csv('Housing.csv')

X , y = house.drop('price' , axis = 1), house['price']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state=26)



ohe = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')
#2 imputer one for categorical for numerical
#and 2 ct, 1 for imputer and one for ohe


ct = make_column_transformer((ohe,make_column_selector(dtype_include=object)),
                            ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False).set_output(transform='pandas')

scaler = MinMaxScaler().set_output(transform='pandas')



ks = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
scores=[]
for k in ks:
    knn = KNeighborsRegressor(n_neighbors=k)
    pipeKNN = Pipeline([('ct',ct),('Scaler',scaler),('KNN',knn)])
    pipeKNN.fit(X_train,y_train)
    y_pred = pipeKNN.predict(X_test)
    scores.append([k,r2_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['K','score'])    
df_scores.sort_values('score',ascending=False)

,K,score
3,4,0.648484
5,6,0.634816
4,5,0.629438
2,3,0.617526
6,7,0.615586
8,9,0.613230
7,8,0.609255
9,10,0.609119
10,11,0.600157
14,15,0.587456
